In [3]:
# ==============================================================================
# ASSIGNMENT 1: UBER FARE PREDICTION
# ==============================================================================
# ALGORITHM & OBJECTIVE:
#   Predict continuous fare amount using Pickup/Dropoff coordinates and timestamp.
#   1. Data Cleaning: Drop nulls, parse datetime, calculate geodesic distance via
#      Haversine formula.
#   2. Outlier Removal: Filter invalid coordinates (lat [-90,90], lon [-180,180]),
#      unrealistic fares (fare > 0 and fare < 500), and extreme distances (< 100km).
#   3. Correlation Analysis: Compute Pearson correlation matrix and visualize.
#   4. Modeling: Fit Ordinary Least Squares (Linear Regression) and Ensemble
#      Decision Trees (Random Forest Regressor).
#   5. Evaluation: Compare RMSE (Root Mean Squared Error) and R² Score.
#
# TIME & SPACE COMPLEXITIES:
#   - Haversine Distance computation: O(N) time, O(1) auxiliary space.
#   - Linear Regression:
#       * Training: O(N * D^2 + D^3) time, where N = samples, D = features.
#       * Inference: O(D) time per sample.
#   - Random Forest Regressor:
#       * Training: O(M * N_sub * D * log(N_sub)), where M = n_estimators.
#       * Inference: O(M * depth) time per sample.
#       * Space: O(M * total_nodes) to store ensemble decision trees.
# ==============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# ---------------------------------------------------------
# Step 1: Pre-process the dataset
# ---------------------------------------------------------
df = pd.read_csv('uber.csv')

# Drop unneeded identifier columns and missing values
df = df.drop(columns=['Unnamed: 0', 'key'], errors='ignore')
df.dropna(inplace=True)

# Parse pickup datetime to extract time components
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
df['hour'] = df['pickup_datetime'].dt.hour
df['day'] = df['pickup_datetime'].dt.day
df['month'] = df['pickup_datetime'].dt.month
df['dayofweek'] = df['pickup_datetime'].dt.dayofweek
df.drop(columns=['pickup_datetime'], inplace=True)

# Vectorized Haversine distance formula (in Kilometers)
def haversine_np(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    km = 6371.0 * c
    return km

df['distance_km'] = haversine_np(
    df['pickup_longitude'].values, df['pickup_latitude'].values,
    df['dropoff_longitude'].values, df['dropoff_latitude'].values
)

# ---------------------------------------------------------
# Step 2: Identify and treat outliers
# ---------------------------------------------------------
# Geographic bounding filters (around NYC coordinates) and valid metrics
df = df[
    (df['fare_amount'] > 2.5) & (df['fare_amount'] < 250) &
    (df['passenger_count'] > 0) & (df['passenger_count'] <= 6) &
    (df['distance_km'] > 0.1) & (df['distance_km'] < 100) &
    (df['pickup_latitude'].between(-90, 90)) &
    (df['dropoff_latitude'].between(-90, 90)) &
    (df['pickup_longitude'].between(-180, 180)) &
    (df['dropoff_longitude'].between(-180, 180))
]

# ---------------------------------------------------------
# Step 3: Check Correlation
# ---------------------------------------------------------
corr_matrix = df.corr()
print("Correlation with fare_amount:\n", corr_matrix['fare_amount'].sort_values(ascending=False))

# ---------------------------------------------------------
# Step 4: Model Training (Linear Regression & Random Forest)
# ---------------------------------------------------------
features = ['distance_km', 'passenger_count', 'hour', 'day', 'month', 'dayofweek']
X = df[features]
y = df['fare_amount']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model A: Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

# Model B: Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=50, max_depth=12, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

# ---------------------------------------------------------
# Step 5: Evaluate Models and Compare Scores
# ---------------------------------------------------------
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

eval_df = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest Regressor'],
    'RMSE': [rmse_lr, rmse_rf],
    'R2_Score': [r2_lr, r2_rf]
})
print("\n--- Model Evaluation Summary ---")
print(eval_df)

C:\Users\Subhi Jain\AppData\Local\Temp\ipykernel_24624\523207060.py:42: Pandas4Warning: The inplace keyword in DataFrame.dropna is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.dropna(inplace=True)
C:\Users\Subhi Jain\AppData\Local\Temp\ipykernel_24624\523207060.py:49: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df['dayofweek'] = df['pickup_datetime'].dt.dayofweek
C:\Users\Subhi Jain\AppData\Local\Temp\ipykernel_24624\523207060.py:50: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns=['pickup_datetime'], inplace=True)


Correlation with fare_amount:
 fare_amount          1.000000
distance_km          0.868532
month                0.024527
passenger_count      0.013048
pickup_longitude     0.012316
dropoff_longitude    0.010784
dayofweek            0.005061
day                 -0.000584
dropoff_latitude    -0.008882
pickup_latitude     -0.009076
hour                -0.019970
Name: fare_amount, dtype: float64

--- Model Evaluation Summary ---
                     Model      RMSE  R2_Score
0        Linear Regression  4.438389  0.781389
1  Random Forest Regressor  3.967542  0.825311
